# Is carrier concentration hiding in ChemDataExtractor after all?

Both notebooks so far checked for a `carrier_concentration`/`hall_carrier_conc`-style **column** in
ChemDataExtractor's schema and found none. But that database stores hundreds of different property types
all inside one free-text `Specifier` column (ZT, thermal conductivity, Seebeck coefficient, ...) -- we've
only ever looked at the top ~10 of 277 distinct values it contains. This notebook checks the other ~267
directly for anything carrier-concentration or Hall-coefficient related, which a column-name search would
never catch.

Run top to bottom.


## 0. Setup


In [1]:
!pip install -q pandas requests pymatgen

import requests
import pandas as pd
import numpy as np
import io
import re
import zipfile
import ast
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)
pd.set_option('display.max_rows', 300)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.1/829.1 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 8.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.

## 1. Load the ChemDataExtractor thermoelectric DB (zip-based, as verified in earlier notebooks)


In [2]:
fig_api = 'https://api.figshare.com/v2/articles/19658787'
resp = requests.get(fig_api, timeout=30)
article = resp.json()
files = article.get('files', [])
zip_candidates = [f for f in files if f['name'].lower().endswith('.zip')]

def load_csvs_from_zip_url(url, timeout=120):
    r = requests.get(url, timeout=timeout)
    zf = zipfile.ZipFile(io.BytesIO(r.content))
    tables = {}
    for n in zf.namelist():
        if n.lower().endswith('.csv'):
            with zf.open(n) as fh:
                try:
                    tables[n] = pd.read_csv(fh)
                except Exception as e:
                    print(n, 'FAILED to parse:', e)
    return tables

cde_tables = {}
te_db_zip = next((f for f in zip_candidates if 'TE_databases' in f['name']), None)
if te_db_zip:
    cde_tables = load_csvs_from_zip_url(te_db_zip['download_url'])
if not cde_tables:
    other_zip = next((f for f in zip_candidates if f is not te_db_zip), None)
    if other_zip:
        cde_tables = load_csvs_from_zip_url(other_zip['download_url'])

df_cde = None
if cde_tables:
    main_key = max(cde_tables, key=lambda k: cde_tables[k].shape[0])
    df_cde = cde_tables[main_key]
    print('Using', main_key, ':', df_cde.shape)

specifier_col = next((c for c in df_cde.columns if c.lower() == 'specifier'), None)
value_col = next((c for c in df_cde.columns if c.lower() == 'value'), None)
units_col = next((c for c in df_cde.columns if c.lower() == 'units'), None)
name_col = next((c for c in df_cde.columns if c.lower() == 'name'), None)
resolution_col = next((c for c in df_cde.columns if 'resolution' in c.lower()), None)
temp_col = next((c for c in df_cde.columns if c.lower() == 'temperature_average'), None)
doi_col = next((c for c in df_cde.columns if c.lower() == 'doi'), None)
print('Key columns:', specifier_col, value_col, units_col, name_col, resolution_col, temp_col, doi_col)


Using TE_databases/CSV/main_tedb.csv : (22805, 24)
Key columns: Specifier Value Units Name Resolution Temperature_Average DOI


## 2. Print every distinct Specifier value, not just the top few

This is the actual check. `Specifier` is free text pulled from paper sentences by ChemDataExtractor's NLP,
so there's no fixed vocabulary -- the only way to know if carrier concentration is in there is to look at
literally all of it.


In [3]:
specifier_counts = df_cde[specifier_col].value_counts()
print('Total distinct Specifier values:', len(specifier_counts))
print()
print('All of them, most common first:')
for spec, cnt in specifier_counts.items():
    print(f'{cnt:6d}  {spec}')


Total distinct Specifier values: 277

All of them, most common first:
  5380  ZT
  1463  thermal conductivity
  1342  figure of merit
  1340  maximum ZT
  1266  Seebeck coefficient
  1072  zT
  1022  power factor
   892  electrical conductivity
   739  highest ZT
   603  S
   603  lattice thermal conductivity
   600  resistivity
   519  peak ZT
   429  σ
   384  κ
   360  conductivity
   293  maximum zT
   270  PF
   251  maximum power factor
   243  κL
   219  thermopower
   188  Figure of merit
   188  highest power factor
   183  κlat
   155  thermal conductivities
   138  peak zT
   129  power factors
   113  highest zT
   105  maximum figure of merit
   105  κl
    86  ρ
    83  α
    83  thermoelectric power
    77  κtot
    76  average ZT
    71  S2σ
    69  figures of merit
    67  highest figure of merit
    65  κtotal
    64  ionic conductivity
    62  maximum Seebeck coefficient
    61  maximum PF
    52  electrical conductivities
    50  highest Seebeck coefficient
    49  

## 3. Keyword search for anything carrier-concentration or Hall-related

Same keyword logic already used (and already confirmed empty) against the column *names* in the earlier
notebooks -- here applied against the Specifier *values* instead, which is the check that was actually
missing.


In [4]:
carrier_keywords = r'carrier|hall|doping|dopant conc|charge conc|free electron|free hole|\bn-type conc|\bp-type conc'
matches = specifier_counts[specifier_counts.index.str.contains(carrier_keywords, case=False, regex=True, na=False)]

print('Specifier values matching carrier/Hall/doping-concentration keywords:')
if len(matches):
    print(matches)
else:
    print('none found -- carrier/Hall concentration is not present anywhere in the Specifier column either.')


Specifier values matching carrier/Hall/doping-concentration keywords:
none found -- carrier/Hall concentration is not present anywhere in the Specifier column either.


## 4. If anything matched: inspect it directly, and check overlap with the cleaned DFT<->experiment formulas

Reuses the cleaned composition-matching filter (D/T-isotope + abbreviation blocklist) from the earlier
notebooks so the result is directly comparable to the overlap counts already established.


In [5]:
if len(matches):
    carrier_rows = df_cde[df_cde[specifier_col].isin(matches.index)].copy()
    print('Rows with a carrier/Hall-concentration-like Specifier:', len(carrier_rows))
    display(carrier_rows[[c for c in [name_col, resolution_col, specifier_col, value_col, units_col, temp_col, doi_col] if c]].head(30))
else:
    carrier_rows = pd.DataFrame()
    print('Nothing to inspect -- see Section 3.')


Nothing to inspect -- see Section 3.


In [6]:
if len(carrier_rows):
    from pymatgen.core.composition import Composition

    NON_FORMULA_LABELS = {'HH', 'SKD', 'M2Q', 'ABQ2', 'Cu2Q', 'clathrate', 'etc', 'CNT'}
    ISOTOPE_SYMBOLS = {'D', 'T'}

    def to_reduced_formula(s, exclude_labels=NON_FORMULA_LABELS):
        if not isinstance(s, str) or not s.strip():
            return None
        if s.strip() in exclude_labels:
            return None
        try:
            comp = Composition(s)
            if any(el.symbol in ISOTOPE_SYMBOLS for el in comp.elements):
                return None
            return comp.reduced_formula
        except Exception:
            return None

    def extract_host(s):
        if not isinstance(s, str):
            return None
        m = re.search(r"host['\"]?\s*[:=]\s*['\"]?([A-Za-z0-9().]+)", s)
        return m.group(1) if m else None

    host_series = carrier_rows[resolution_col].map(extract_host) if resolution_col else None
    if host_series is None or host_series.notna().sum() == 0:
        host_series = carrier_rows[name_col]
    carrier_rows['host_formula'] = host_series.map(to_reduced_formula)

    print('Carrier/Hall-concentration rows with a parseable host formula:', carrier_rows['host_formula'].notna().sum(), '/', len(carrier_rows))
    print()
    print('Distinct host formulas found (cross-check these against your existing overlap_cde set by hand):')
    print(sorted(carrier_rows['host_formula'].dropna().unique().tolist()))
else:
    print('Nothing to check -- Section 3 found no carrier/Hall-concentration rows at all.')


Nothing to check -- Section 3 found no carrier/Hall-concentration rows at all.


## 5. Verdict

- **If Section 3 found nothing**: confirmed -- carrier/Hall concentration is genuinely absent from
  ChemDataExtractor, not just missing as a column. This closes off option 1 from the discussion and means
  the conductivity-matching approach (deriving n from measured resistivity + the DFT conductivity-vs-n
  curve) is the next thing worth building, since it's the only fully algorithmic option left.
- **If Section 3 found something**: check Section 4's host-formula list against the already-confirmed
  composition overlap (the ~9 teMatDb272 / ~41 ChemDataExtractor formulas from the earlier notebooks) --
  only rows on a formula that's *also* in that overlap set are directly useful for computing the residual
  against the same DFT entries already being used.
